#### load the data

In [ ]:
import json
import os
import random

import pandas as pd
from datasets import load_dataset

ds = load_dataset("AGBonnet/augmented-clinical-notes", split="train")
print(ds)

In [ ]:
row = ds[0]
print(row.keys())
print(row["note"][:1000])

#### see the summary’s structure

In [ ]:
summary = json.loads(row["summary"])
print(summary.keys())
print(json.dumps(summary, indent=2)[:2000])

#### list all summary sections

In [ ]:
for key in summary:
    print(key)

#### show only the medication/treatment section

In [ ]:
for key in summary:
    if "treat" in key.lower() or "medic" in key.lower():
        print("==", key, "==")
        print(json.dumps(summary[key], indent=2))

#### medication fields across 40 notes

In [ ]:
from collections import Counter

# Some summaries contain raw line breaks/tabs inside text, which strict JSON rejects.
# strict=False tolerates them; anything still broken is recorded instead of crashing.
bad_records = []

def safe_load(i):
    try:
        return json.loads(ds[i]["summary"], strict=False)
    except json.JSONDecodeError:
        bad_records.append(i)
        return None

def present(v):
    return isinstance(v, str) and v.strip().lower() not in {"", "none", "n/a", "unknown", "not specified"}

# "treatments" is not always a list of dicts: it can be a single dict, a string, or a list mixing strings.
shape_counts = Counter()

def get_treatments(s):
    tr = s.get("treatments")
    if isinstance(tr, dict):
        shape_counts["single dict"] += 1
        return [tr]
    if isinstance(tr, list):
        items = [t for t in tr if isinstance(t, dict)]
        if len(items) < len(tr):
            shape_counts["list with non-dict items"] += 1
        else:
            shape_counts["list of dicts"] += 1
        return items
    shape_counts[f"other: {type(tr).__name__}"] += 1
    return []

random.seed(42)
idxs = random.sample(range(len(ds)), 40)

STATUS_WORDS = ["start", "stop", "discontinu", "switch", "chang", "increas", "decreas", "taper", "withdr", "resum", "replac"]
ROUTE_WORDS = ["oral", "intravenous", "iv ", "subcutaneous", "intramuscular", "topical", "inhal", "per os"]

rows = []
for i in idxs:
    s = safe_load(i)
    if s is None:
        continue
    for t in get_treatments(s):
        if not present(t.get("name")):
            continue
        text = " ".join(str(t.get(k, "")) for k in ["dosage", "details", "time", "frequency"]).lower()
        rows.append({
            "note_idx": i,
            "name": t.get("name"),
            "has_dosage": present(t.get("dosage")),
            "has_frequency": present(t.get("frequency")),
            "has_duration": present(t.get("duration")),
            "route_hint": any(w in text for w in ROUTE_WORDS),
            "status_hint": any(w in text for w in STATUS_WORDS),
        })

df = pd.DataFrame(rows)
print("notes sampled:", len(idxs))
print("unparseable summaries:", len(set(bad_records)), "/ 40")
print("treatments field shapes:", dict(shape_counts))
print("treatments found:", len(df))
print("notes with at least 1 treatment:", df["note_idx"].nunique())
print(df[["has_dosage", "has_frequency", "has_duration", "route_hint", "status_hint"]].mean().round(2))

In [ ]:
lengths, age_ok, sex_ok = [], 0, 0
for i in idxs:
    rec = ds[i]
    s = safe_load(i)
    p = s.get("patient information", {}) if s else {}
    if not isinstance(p, dict):
        p = {}
    lengths.append(len(rec["note"].split()))
    age_ok += present(p.get("age"))
    sex_ok += present(p.get("sex"))

print("note length (words): min", min(lengths), "| median", sorted(lengths)[20], "| max", max(lengths))
print("age present:", age_ok, "/ 40")
print("sex present:", sex_ok, "/ 40")

In [ ]:
full_lengths = sorted(len(ds[i]["full_note"].split()) for i in idxs)
print("full_note length (words): min", full_lengths[0], "| median", full_lengths[20], "| max", full_lengths[-1])

same = sum(ds[i]["note"].strip() == ds[i]["full_note"].strip() for i in idxs)
print("note identical to full_note:", same, "/ 40")

In [ ]:
print(df.sample(10, random_state=1)[["name", "has_dosage", "has_frequency", "route_hint", "status_hint"]].to_string())

## Other datasets: viability check
### MedQuAD

In [ ]:
mq = load_dataset("lavita/MedQuAD", split="train")
print(mq)
print(mq[0])

In [ ]:
import collections

print(mq.column_names)

answers = mq["answer"]
has_answer = sum(1 for a in answers if a and str(a).strip())
print("rows with an answer:", has_answer, "/", len(mq))


print("by source (with answers):", collections.Counter(
    s for s, a in zip(mq["document_source"], answers) if a and str(a).strip()
))
print("question types:", collections.Counter(mq["question_type"]).most_common(10))

In [ ]:
terms = ["craniotomy", "edema", "hypertension", "anemia", "sepsis",
         "biopsy", "stent", "dialysis", "seizure", "pneumonia",
         "intubation", "embolism", "carcinoma", "arrhythmia", "catheter"]

focus = [str(f).lower() for f, a in zip(mq["question_focus"], mq["answer"]) if a and str(a).strip()]
for t in terms:
    hits = sum(t in f for f in focus)
    print(f"{t:15} {hits}")

### MeSH

In [ ]:
import requests

BASE = "https://id.nlm.nih.gov/mesh"

def find_descriptor(term):
    for match in ["exact", "startsWith", "contains"]:
        r = requests.get(f"{BASE}/lookup/descriptor",
                         params={"label": term, "match": match, "limit": 1}, timeout=20)
        if r.ok and r.json():
            hit = r.json()[0]
            return hit["label"], hit["resource"].rsplit("/", 1)[-1]
    return None, None

def scope_note(ui):
    d = requests.get(f"{BASE}/{ui}.json", timeout=20).json()
    concept = d.get("preferredConcept")
    if not concept:
        return "(no preferred concept found)"
    c = requests.get(concept.replace("http://", "https://") + ".json", timeout=20).json()
    note = c.get("scopeNote")
    if isinstance(note, dict):
        note = note.get("@value")
    return note or "(no scope note)"

for term in ["craniotomy", "intubation", "catheter", "biopsy", "sepsis", "stent", "perilesional edema"]:
    label, ui = find_descriptor(term)
    if not ui:
        print(f"{term:20} -> NOT FOUND\n")
        continue
    print(f"{term:20} -> {label} ({ui})")
    print("   ", str(scope_note(ui))[:250], "\n")

### Noisy Medical Document Images

In [ ]:
import kagglehub

path = kagglehub.dataset_download("devp1866/noisy-medical-document-images-ocr")
print("downloaded to:", path)

for root, dirs, files in os.walk(path):
    depth = root.replace(path, "").count(os.sep)
    if depth <= 2:
        print("  " * depth + os.path.basename(root) + "/", f"({len(files)} files)")

In [ ]:
data_dir = os.path.join(path, "Data")
print("files in Data/:", [f for f in os.listdir(data_dir) if os.path.isfile(os.path.join(data_dir, f))])

ds_dir = os.path.join(data_dir, "discharge_summaries")
names = sorted(os.listdir(ds_dir))
print("first 5 discharge files:", names[:5])
print("file types:", {os.path.splitext(n)[1] for n in names})

In [ ]:
from IPython.display import Image, display

first_image = os.path.join(ds_dir, names[0])
display(Image(filename=first_image, width=600))

In [ ]:
gt = pd.read_csv(os.path.join(data_dir, "discharge_summaries_ground_truth.csv"))
print(gt.shape)
print(gt.columns.tolist())
print(gt.iloc[0].to_dict())

In [ ]:
import re

docs = [json.loads(j) for j in gt["json_data"]]

meds = [m for d in docs for m in d.get("medications", [])]
print("documents:", len(docs))
print("unique hospitals:", len({d["hospital"]["name"] for d in docs}))
print("unique diagnoses:", len({d.get("diagnosis") for d in docs}))
print("medications total:", len(meds), "| unique:", len(set(meds)))
print("meds per document: min", min(len(d.get("medications", [])) for d in docs),
      "| max", max(len(d.get("medications", [])) for d in docs))
print("with route (PO/IV/IM/SC/topical):", round(sum(bool(re.search(r"\b(PO|IV|IM|SC|SubQ|Topical|Inhal)", m, re.IGNORECASE)) for m in meds) / len(meds), 2))
print("with a dose (mg/mcg/units/mL):", round(sum(bool(re.search(r"\d+\s?(mg|mcg|g|units|mL)", m, re.IGNORECASE)) for m in meds) / len(meds), 2))
print("top diagnoses:", collections.Counter(d.get("diagnosis") for d in docs).most_common(5))

### ClinOCR-Bench

In [ ]:
subsets = ["normal", "handwriting", "poor", "rotated", "tables", "mixed"]
clin = {}
for s in subsets:
    clin[s] = load_dataset("ClinOCR-Bench/ClinOCR-Bench", s)
    print(f"{s:12} train={len(clin[s]['train'])} test={len(clin[s]['test'])}")

print("columns:", clin["normal"]["test"].column_names)

In [ ]:
ex = clin["mixed"]["test"][0]
for k, v in ex.items():
    if k != "image":
        print(k, ":", str(v)[:300])
display(ex["image"].resize((600, int(600 * ex["image"].height / ex["image"].width))))

In [ ]:
ex = clin["mixed"]["test"][0]
print("ground_truth (first 600 chars):\n", ex["ground_truth"][:600])
print("\nground_truth length (words):", len(ex["ground_truth"].split()))

total = sum(len(clin[s]["train"]) + len(clin[s]["test"]) for s in subsets)
templates = {t for s in subsets for split in ["train", "test"] for t in clin[s][split]["template"]}
print("\ntotal images:", total)
print("unique templates:", len(templates))
for s in subsets:
    print(f"{s:12} train={len(clin[s]['train'])} test={len(clin[s]['test'])}")

### RxNorm and openFDA

In [ ]:

for drug in ["Meloxicam", "Hydroxychloroquine", "Ibuprofen", "Belimumab", "Ferrous Sulfate"]:
    r = requests.get("https://rxnav.nlm.nih.gov/REST/rxcui.json", params={"name": drug}, timeout=20).json()
    ids = r.get("idGroup", {}).get("rxnormId", [])
    f = requests.get("https://api.fda.gov/drug/label.json",
                     params={"search": f'openfda.generic_name:"{drug}"', "limit": 1}, timeout=20)
    label = f.json().get("results", [{}])[0] if f.ok else {}
    purpose = (label.get("indications_and_usage") or label.get("purpose") or ["(no label found)"])[0]
    print(f"{drug:20} RxCUI={ids[:1] or 'NOT FOUND'}")
    print("   FDA label:", purpose[:200], "\n")

In [ ]:
URL = "https://api.fda.gov/drug/label.json"

def fda_label(drug, route=None):
    queries = [
        f'openfda.generic_name:"{drug}" AND openfda.product_type:"HUMAN PRESCRIPTION DRUG"',
        f'openfda.brand_name:"{drug}" AND openfda.product_type:"HUMAN PRESCRIPTION DRUG"',
        f'"{drug}" AND openfda.product_type:"HUMAN PRESCRIPTION DRUG"',
    ]
    for q in queries:
        if route:
            q += f' AND openfda.route:"{route}"'
        r = requests.get(URL, params={"search": q, "limit": 5}, timeout=20)
        if not r.ok:
            continue
        for lab in r.json().get("results", []):
            text = " ".join(lab.get("indications_and_usage", []) + lab.get("purpose", []))
            if text and "homeopathic" not in text.lower():
                return q.split(" AND ")[0], text
    return None, "(no reliable label found)"

tests = [("Meloxicam", "ORAL"), ("Hydroxychloroquine", "ORAL"), ("Ibuprofen", "ORAL"),
         ("Belimumab", None), ("Ferrous Sulfate", "ORAL")]

for drug, route in tests:
    q, text = fda_label(drug, route)
    print(f"{drug:20} matched via: {q}")
    print("   ", text[:200], "\n")

In [ ]:
import requests

for drug in ["meloxicam", "hydroxychloroquine", "ibuprofen", "belimumab", "ferrous sulfate"]:
    r = requests.get("https://rxnav.nlm.nih.gov/REST/rxclass/class/byDrugName.json",
                     params={"drugName": drug}, timeout=20).json()
    items = r.get("rxclassDrugInfoList", {}).get("rxclassDrugInfo", [])
    classes = sorted({(i["rela"], i["relaSource"], i["rxclassMinConceptItem"]["className"]) for i in items})
    print(drug)
    for rela, src, name in classes[:8]:
        print(f"   {src:8} {rela:20} {name}")
    print()

In [ ]:
for drug in ["meloxicam", "hydroxychloroquine", "ibuprofen", "belimumab", "ferrous sulfate"]:
    r = requests.get("https://rxnav.nlm.nih.gov/REST/rxclass/class/byDrugName.json",
                     params={"drugName": drug}, timeout=20).json()
    items = r.get("rxclassDrugInfoList", {}).get("rxclassDrugInfo", [])
    epc = sorted({i["rxclassMinConceptItem"]["className"] for i in items if i["rela"] == "has_epc"})
    print(f"{drug:20} {epc or 'NO EPC FOUND'}")

In [ ]:
r = requests.get("https://rxnav.nlm.nih.gov/REST/rxclass/class/byDrugName.json",
                 params={"drugName": "ferrous sulfate"}, timeout=20)
print("status:", r.status_code)
items = r.json().get("rxclassDrugInfoList", {}).get("rxclassDrugInfo", [])

by_type = {}
for i in items:
    c = i["rxclassMinConceptItem"]
    by_type.setdefault(c["classType"], set()).add(c["className"])

for ctype, names in sorted(by_type.items()):
    print(f"{ctype:12} {sorted(names)[:6]}")

In [ ]:
from huggingface_hub import login

login()

In [ ]:
for name in ["openlanguagedata/flores_plus", "ai4bharat/IN22-Conv"]:
    try:
        d = load_dataset(name)
        print(name, "->", d)
    except Exception as e:
        print(name, "-> ERROR:", type(e).__name__, str(e)[:300])
    print()

In [ ]:
import collections

conv = load_dataset("ai4bharat/IN22-Conv")["test"]
print(collections.Counter(conv["domain"]).most_common(15))

health = [r for r in conv if "health" in str(r["domain"]).lower() or "health" in str(r["topic"]).lower()]
print("health-related sentences:", len(health))
for r in health[:3]:
    print("EN:", r["eng_Latn"])
    print("HI:", r["hin_Deva"], "\n")

In [ ]:
for cfg in ["hin_Deva", "eng_Latn"]:
    try:
        f = load_dataset("openlanguagedata/flores_plus", cfg)
        print(cfg, "->", f)
    except Exception as e:
        print(cfg, "-> ERROR:", type(e).__name__, str(e)[:300])

In [ ]:
hi = load_dataset("openlanguagedata/flores_plus", "hin_Deva")["devtest"]
en = load_dataset("openlanguagedata/flores_plus", "eng_Latn")["devtest"]

print("hindi rows:", len(hi), "| english rows:", len(en))
print("ids aligned:", hi["id"] == en["id"])
print("domains:", collections.Counter(en["domain"]).most_common())
print("health-ish topics:", sum("health" in str(t).lower() or "medic" in str(t).lower() for t in en["topic"]))

In [ ]:
import collections
import json
import random
import re

from datasets import load_dataset

ds = load_dataset("AGBonnet/augmented-clinical-notes", split="train")

def safe_load(i):
    try:
        return json.loads(ds[i]["summary"], strict=False)
    except json.JSONDecodeError:
        return None

def present(v):
    return isinstance(v, str) and v.strip().lower() not in {"", "none", "n/a", "unknown", "not specified"}

def get_treatments(s):
    tr = s.get("treatments")
    if isinstance(tr, dict):
        return [tr]
    if isinstance(tr, list):
        return [t for t in tr if isinstance(t, dict)]
    return []

def norm(s):
    return re.sub(r"\s+", " ", s.lower())

random.seed(42)
idxs = random.sample(range(len(ds)), 40)

counts = collections.Counter()
for i in idxs:
    rec = ds[i]
    s = safe_load(i)
    if not s:
        continue
    short, full = norm(rec["note"]), norm(rec["full_note"])
    for t in get_treatments(s):
        name = t.get("name")
        if not present(name):
            continue
        key = norm(name).split()[0]
        if key in short:
            counts["found in note (first ~300 words)"] += 1
        elif key in full:
            counts["only in full_note (after the cut)"] += 1
        else:
            counts["in neither"] += 1
print(counts)

In [ ]:
# Run the rule-based extractor on the 19 unique Kaggle medication lines
import json
import os

import kagglehub
import pandas as pd

from lablens.extract_rules import extract

path = kagglehub.dataset_download("devp1866/noisy-medical-document-images-ocr")
gt = pd.read_csv(os.path.join(path, "Data", "discharge_summaries_ground_truth.csv"))
lines = sorted({m for j in gt["json_data"] for m in json.loads(j)["medications"]})
for line in lines:
    meds = extract(line)
    print(f"{line:55} -> {[(m.drug, m.strength, m.dosage, m.route, m.frequency) for m in meds]}")